# 🔬 InceptionV3 - Interprétabilité avec Grad-CAM

Ce notebook charge un modèle **InceptionV3 pré-entraîné** et analyse ses prédictions avec Grad-CAM.

## 📦 Utilise les 5 modules refactorisés

- `data_utils` - Chargement et preprocessing (224x224 pour InceptionV3)
- `model_builders` - Pas utilisé (modèle déjà entraîné)
- `training_utils` - Évaluation uniquement
- `visualization_utils` - Graphiques
- `interpretability_utils` - Grad-CAM

## 1. Configuration et Imports

In [ ]:
"""
╔════════════════════════════════════════════════════════════════════════════╗
║  🎯 CELLULE DE CONFIGURATION STANDALONE - COPIER-COLLER DANS VOS NOTEBOOKS ║
╚════════════════════════════════════════════════════════════════════════════╝

INSTRUCTIONS:
-------------
1. Copiez TOUT le contenu de cette cellule
2. Collez-le comme PREMIÈRE CELLULE de votre notebook
3. Exécutez la cellule
4. Les variables sont prêtes à l'emploi !

Cette cellule est 100% autonome et fonctionne partout :
✅ Google Colab (clone + installe automatiquement)
✅ WSL / Linux Local
✅ Tout environnement Jupyter

APRÈS EXÉCUTION, VOUS POUVEZ UTILISER:
- config: Objet de configuration (config.batch_size, config.data_dir, etc.)
- ENV: Environnement détecté ('colab', 'wsl', 'local')
- Tous les imports des transformers

"""

# =============================================================================
# IMPORTS STANDARDS
# =============================================================================

import os
import sys
import subprocess
from pathlib import Path


# =============================================================================
# DÉTECTION AUTOMATIQUE DE L'ENVIRONNEMENT
# =============================================================================

def detect_environment():
    """Détecte l'environnement (colab, wsl, local)"""
    try:
        import google.colab
        return "colab"
    except ImportError:
        is_wsl = os.path.exists('/proc/version') and 'microsoft' in open('/proc/version').read().lower()
        return "wsl" if is_wsl else "local"

ENV = detect_environment()
print(f"🌍 Environnement: {ENV.upper()}")


# =============================================================================
# BOOTSTRAP COLAB (Clone + Install si nécessaire)
# =============================================================================

if ENV == "colab":
    print("\n🚀 Bootstrap Colab...")
    
    os.chdir('/content')
    if not os.path.exists('/content/Data_Pipeline'):
        print("📥 Clonage du repository...")
        subprocess.run(['git', 'clone', 'https://github.com/L-Poca/Data_Pipeline.git'], check=True)
    
    os.chdir('/content/Data_Pipeline')
    
    # Checkout de la branche rafael_cleaning
    result = subprocess.run(
        ['git', 'checkout', '-b', 'rafael_cleaning', 'origin/rafael_cleaning'],
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        # Si la branche locale existe déjà, juste switcher
        subprocess.run(['git', 'checkout', 'rafael_cleaning'], capture_output=True)
    
    # Installation du package en mode éditable (sans dépendances - détection Colab dans setup.py)
    print("📦 Installation du package...")
    result = subprocess.run(['pip', 'install', '-e', '.', '--quiet'], capture_output=True, text=True)
    if result.returncode != 0:
        print(f"⚠️ Erreur installation: {result.stderr}")
    else:
        print("✅ Package installé")
    
    print("💾 Montage Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Extraction dataset
    archive_data = '/content/drive/MyDrive/DS_COVID/archive_covid.zip'
    if os.path.exists(archive_data):
        print("📦 Extraction dataset...")
        os.makedirs('./data/raw/', exist_ok=True)
        subprocess.run(['unzip', '-o', '-q', archive_data, '-d', './data/raw/COVID-19_Radiography_Dataset/'])
    
    # Extraction models
    archive_models = '/content/drive/MyDrive/DS_COVID/inceptionv3_best.zip'
    if os.path.exists(archive_models):
        print("📦 Extraction models...")
        os.makedirs('./models/', exist_ok=True)
        subprocess.run(['unzip', '-o', '-q', archive_models, '-d', './models/'])

    print("✅ Bootstrap terminé")


# =============================================================================
# CONFIGURATION DES CHEMINS
# =============================================================================

# Déterminer project_root selon l'environnement
if ENV == "colab":
    project_root = Path('/content/Data_Pipeline')
elif ENV == "wsl":
    project_root = Path('/home/cepa/DST/projet_DS/Data_Pipeline/Data_Pipeline')
else:  # local
    # Depuis un notebook dans src/notebooks/
    project_root = Path.cwd().parent.parent

# Vérification du modèle en local (WSL ou autre)
if ENV != "colab":
    models_dir = project_root / 'models'
    model_path = models_dir / '1345_masked/inceptionv3_best.keras'
    
    if model_path.exists():
        print(f"✅ Modèle InceptionV3 trouvé: {model_path}")
    else:
        print(f"⚠️ Modèle InceptionV3 non trouvé: {model_path}")
        print(f"   Veuillez placer inceptionv3_best.keras dans {models_dir}/")

# Ajouter src/ au sys.path pour les imports
# src_path = str(project_root / 'src')
# if src_path not in sys.path:
#     sys.path.insert(0, src_path)
#     print(f"✅ Chemin src/ ajouté: {src_path}")

# Charger la configuration depuis JSON
from src.utils.config import build_config

config = build_config(project_root, ENV)

# Exports pour compatibilité avec anciens notebooks
data_dir = config.data_dir
categories = config.classes
img_size = config.img_size


# =============================================================================
# IMPORTS DES TRANSFORMERS
# =============================================================================

try:
    from src.features.Pipelines.Transformateurs.image_loaders import ImageLoader
    from src.features.Pipelines.Transformateurs.image_preprocessing import (
        ImageResizer, ImageNormalizer, ImageFlattener, ImageMasker
    )
    from src.features.Pipelines.Transformateurs.image_augmentation import (
        ImageAugmenter, ImageRandomCropper
    )
    from src.features.Pipelines.Transformateurs.image_features import (
        ImageHistogram, ImagePCA, ImageStandardScaler
    )
    print("✅ Transformers importés")
except ImportError as e:
    print(f"⚠️ Erreur import transformers: {e}")


# =============================================================================
# IMPORTS ML/DL
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras

# =============================================================================
# CONFIGURATION MATPLOTLIB
# =============================================================================

plt.rcParams['figure.figsize'] = (15, 10)
sns.set_style('whitegrid')

# =============================================================================
# AFFICHAGE DU RÉSUMÉ
# =============================================================================

print("\n" + "=" * 70)
print("✅ CONFIGURATION PRÊTE - Data Pipeline")
print("=" * 70)
print(f"📂 Projet: {project_root}")
print(f"📊 Dataset: {data_dir}")
print(f"🏷️ Classes: {', '.join(categories)}")
print(f"🎛️ Images: {img_size}")
print(f"🔧 Batch: {config.batch_size} | Époques: {config.epochs}")
print(f"📐 Dataset accessible: {'✅' if data_dir.exists() else '❌'}")
print("=" * 70)
print("\n💡 Variables disponibles:")
print("   • config: Configuration complète (Config object)")
print("   • ENV: Environnement actuel")
print("\n🎯 Transformers disponibles:")
print("   • ImageLoader, ImageResizer, ImageNormalizer, ImageFlattener")
print("   • ImageAugmenter, ImageRandomCropper")
print("   • ImageHistogram, ImagePCA, ImageStandardScaler")
print("=" * 70)


In [ ]:
# Imports des fonctions utilitaires depuis les 5 nouveaux modules
from src.notebooks import (
    # Data utils
    load_dataset,
    create_preprocessing_pipeline,
    prepare_train_val_test_split,
    create_transfer_learning_generators,  # Pour preprocessing InceptionV3

    # Training utils (évaluation uniquement)
    evaluate_model,

    # Visualization utils
    plot_confusion_matrix,

    # Interpretability utils
    setup_interpretability,
    select_sample_images,
    run_gradcam_analysis,
    get_preprocessing_function,  # Pour appliquer le preprocessing InceptionV3 dans Grad-CAM
)

print("✅ Fonctions utilitaires importées")
print("\n📦 Ce notebook utilise:")
print("   • Modèle pré-entraîné: InceptionV3")
print("   • Images: 224x224 (requis pour InceptionV3)")
print("   • Preprocessing: InceptionV3-specific")

In [ ]:
# Définition des paramètres principaux
VERBOSE = True
LOAD_MASKS = True  # True pour version maskée
N_IMAGES_PER_CLASS = 100 # None = Charger toutes les images
TEST_SIZE = 0.15
VAL_SIZE = 0.15
BATCH_SIZE = 32  # Plus petit pour InceptionV3 (modèle plus lourd)
RANDOM_SEED = 42

# Chemin du modèle pré-entraîné
MODEL_PATH = config.models_dir / 'inceptionv3_best.keras'

## 2. Chargement et Préparation des Données

In [ ]:
# ⚠️ NOUVEAU: load_dataset retourne 4 valeurs au lieu de 2
# (image_paths, mask_paths, labels, labels_int)
image_paths, mask_paths, labels, labels_int = load_dataset(
    data_dir=config.data_dir,
    categories=config.classes,
    n_images_per_class=N_IMAGES_PER_CLASS,  # None = Charger toutes les images
    load_masks=LOAD_MASKS,  # True pour version maskée
    verbose=VERBOSE
)

In [ ]:
# Créer la pipeline de preprocessing pour InceptionV3 (224x224)
pipeline = create_preprocessing_pipeline(
    img_size=(224, 224),  # InceptionV3 nécessite 224x224
    color_mode='RGB',
    mask_paths=None,
    verbose=VERBOSE
)

# Charger et préprocesser les images
print("\n🔄 Chargement des images...")
images = pipeline.fit_transform(image_paths)

# ⚠️ NE PAS normaliser ici - le preprocessing InceptionV3 le fera
print(f"\n📊 Images préparées:")
print(f"   Shape: {images.shape}")
print(f"   Range: [{images.min():.1f}, {images.max():.1f}]")
print(f"   Dtype: {images.dtype}")
print(f"\n⚠️ Preprocessing InceptionV3 sera appliqué par les generators")

In [ ]:
# Split train/val/test
x_train, x_val, x_test, y_train_cat, y_val_cat, y_test_cat = prepare_train_val_test_split(
    images=images,
    labels_int=labels_int,
    num_classes=len(config.classes),
    test_size=TEST_SIZE,
    val_size=VAL_SIZE,
    random_seed=RANDOM_SEED,
    verbose=VERBOSE
)

# Récupérer les labels integer pour le calcul des class weights et confusion matrix
y_train = np.argmax(y_train_cat, axis=1)
y_val = np.argmax(y_val_cat, axis=1)
y_test = np.argmax(y_test_cat, axis=1)

In [ ]:
# Créer les data generators avec preprocessing InceptionV3
train_generator, val_generator, test_generator = create_transfer_learning_generators(
    x_train=x_train,
    y_train_cat=y_train_cat,
    x_val=x_val,
    y_val_cat=y_val_cat,
    x_test=x_test,
    y_test_cat=y_test_cat,
    base_model_name='InceptionV3',  # Preprocessing spécifique
    batch_size=BATCH_SIZE,
    augment_train=False,  # Pas d'augmentation (modèle déjà entraîné)
    verbose=VERBOSE
)

## 3. Chargement du Modèle Pré-entraîné

In [ ]:
# Charger le modèle InceptionV3 pré-entraîné
print("=" * 70)
print("CHARGEMENT DU MODÈLE INCEPTIONV3")
print("=" * 70)

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Modèle introuvable: {MODEL_PATH}\n"
        f"Assurez-vous d'avoir extrait inceptionv3_best.zip dans models/"
    )

print(f"\n📂 Chargement depuis: {MODEL_PATH}")
model = keras.models.load_model(MODEL_PATH)

print(f"\n✅ Modèle chargé: {model.name}")
print(f"   Input shape: {model.input_shape}")
print(f"   Output shape: {model.output_shape}")
print(f"   Parameters: {model.count_params():,}")

# Afficher le résumé
model.summary()

## 4. Évaluation du Modèle

In [ ]:
# Évaluer le modèle sur le test set
results = evaluate_model(
    model=model,
    test_data=test_generator,  # Utiliser le generator avec preprocessing InceptionV3
    class_names=config.classes,
    verbose=VERBOSE
)

# Extraire les prédictions
y_pred = results['y_pred']
y_pred_probs = results['y_pred_probs']
y_test = results['y_true']

## 5. Visualisation - Matrice de Confusion

In [ ]:
# Créer le répertoire pour les résultats
plots_dir = config.results_dir / 'inceptionv3_interpretability'
plots_dir.mkdir(parents=True, exist_ok=True)

# Matrice de confusion
plot_confusion_matrix(
    y_true=y_test,
    y_pred=y_pred,
    class_names=config.classes,
    normalize=True,
    figsize=(10, 8),
    save_path=plots_dir / 'confusion_matrix.png'
)
plt.show()

## 6. Interprétabilité - Grad-CAM

In [ ]:
# Setup Grad-CAM pour InceptionV3
gradcam = setup_interpretability(
    model=model,
    verbose=VERBOSE
)

# Récupérer la fonction de preprocessing InceptionV3
preprocess_fn = get_preprocessing_function('InceptionV3')
print(f"\n✅ Preprocessing InceptionV3 configuré pour Grad-CAM")

In [ ]:
# Sélectionner des images échantillons (correctement classifiées)
indices_correct, descriptions_correct = select_sample_images(
    x_data=x_test,
    y_true=y_test,
    y_pred=y_pred,
    class_names=config.classes,
    n_samples=2,  # 2 par classe
    strategy='correct',
    random_seed=RANDOM_SEED
)

print(f"\n📊 {len(indices_correct)} échantillons correctement classifiés sélectionnés")

### 6.1 Grad-CAM sur Prédictions Correctes

In [ ]:
### 6.2 Grad-CAM sur Erreurs de Prédiction

In [ ]:
# Sélectionner quelques échantillons mal classifiés
indices_incorrect, descriptions_incorrect = select_sample_images(
    x_data=x_test,
    y_true=y_test,
    y_pred=y_pred,
    class_names=config.classes,
    n_samples=2,  # 2 par classe si disponible
    strategy='incorrect',
    random_seed=RANDOM_SEED
)

print(f"\n📊 {len(indices_incorrect)} échantillons mal classifiés trouvés")

In [ ]:
# Analyse Grad-CAM sur les échantillons correctement classifiés
interp_dir = plots_dir / 'gradcam_correct'
interp_dir.mkdir(parents=True, exist_ok=True)

run_gradcam_analysis(
    gradcam=gradcam,
    x_data=x_test,
    indices=indices_correct,
    descriptions=descriptions_correct,
    class_names=config.classes,
    y_pred_probs=y_pred_probs,
    save_dir=interp_dir,
    preprocess_fn=preprocess_fn  # ✅ AJOUT: Preprocessing InceptionV3
)

In [ ]:
if len(indices_incorrect) > 0:
    # Analyse Grad-CAM sur les erreurs
    interp_dir_errors = plots_dir / 'gradcam_errors'
    interp_dir_errors.mkdir(parents=True, exist_ok=True)
    
    run_gradcam_analysis(
        gradcam=gradcam,
        x_data=x_test,
        indices=indices_incorrect,
        descriptions=descriptions_incorrect,
        class_names=config.classes,
        y_pred_probs=y_pred_probs,
        save_dir=interp_dir_errors,
        preprocess_fn=preprocess_fn  # ✅ AJOUT: Preprocessing InceptionV3
    )
else:
    print("\n✅ Aucune erreur trouvée (modèle parfait sur le test set!)")

## 7. Résumé

✅ Notebook d'interprétabilité pour **InceptionV3 pré-entraîné** !

### 🎯 Caractéristiques

1. **Modèle pré-entraîné** : Chargement d'InceptionV3 depuis `models/inceptionv3_best.keras`
2. **Images 224x224** : Taille requise pour InceptionV3
3. **Preprocessing spécifique** : Normalisation InceptionV3 ([-1, 1])
4. **Évaluation complète** : Métriques et matrice de confusion
5. **Grad-CAM** : Analyse des prédictions correctes et erreurs
6. **Variables snake_case** : Conformité avec les scripts refactorisés

### 📦 Modules utilisés

```
src/notebooks/
├── data_utils.py              # load_dataset, preprocessing, generators
├── training_utils.py          # evaluate_model
├── visualization_utils.py     # plot_confusion_matrix
└── interpretability_utils.py  # Grad-CAM analysis
```

### 🔍 Résultats générés

- **Matrice de confusion** : Performance globale du modèle
- **Grad-CAM correct** : Visualisation des zones d'attention pour les bonnes prédictions
- **Grad-CAM erreurs** : Analyse des cas d'erreur pour comprendre les faiblesses

In [ ]:
print("=" * 70)
print("🎉 ANALYSE TERMINÉE")
print("=" * 70)
print("\n✅ Modèle InceptionV3 évalué")
print("✅ Matrice de confusion générée")
print("✅ Grad-CAM analysé (correct + erreurs)")
print("\n📊 Résultats sauvegardés dans:")
print(f"   • Plots: {plots_dir}")
print(f"   • Grad-CAM correct: {interp_dir}")
if len(indices_incorrect) > 0:
    print(f"   • Grad-CAM erreurs: {interp_dir_errors}")